In [1]:
!pip -q install streamlit scikit-learn pandas numpy joblib openpyxl

import sys, sklearn, pandas, numpy, joblib, streamlit
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)
print("streamlit:", streamlit.__version__)

Python: 3.13.18
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.4.6
joblib: 1.6.0
streamlit: 1.67.0


In [2]:
from pathlib import Path
import pandas as pd

# Carpeta principal del proyecto (aquí irán todos los archivos)
PROJECT = Path("/content/ml_streamlit_project")
DATA_DIR = PROJECT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Dataset público de IBM: Telco Customer Churn
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Guardamos una copia dentro del proyecto (la usará train.py)
df.to_csv(DATA_DIR / "telco_churn.csv", index=False)

print("Filas y columnas:", df.shape)
df.head()

Filas y columnas: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("=== Tipos de datos ===")
display(df.dtypes)

print("\n=== Valores faltantes por columna ===")
display(df.isna().sum().sort_values(ascending=False).head(10))

print("\n=== Clientes repetidos (customerID duplicado) ===")
print(df["customerID"].duplicated().sum())

=== Tipos de datos ===


,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object



=== Valores faltantes por columna ===


,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0



=== Clientes repetidos (customerID duplicado) ===
0


In [4]:
# ¿Por qué TotalCharges es texto? Buscamos valores que no son números
total_num = pd.to_numeric(df["TotalCharges"], errors="coerce")
problematicos = df[total_num.isna()]

print("Filas con TotalCharges no numérico:", len(problematicos))
display(problematicos[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

Filas con TotalCharges no numérico: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [5]:
print("=== Distribución de Churn (variable objetivo) ===")
display(df["Churn"].value_counts())
display((df["Churn"].value_counts(normalize=True) * 100).round(1))

print("\n=== % de abandono según tipo de contrato ===")
display(
    df.groupby("Contract")["Churn"]
      .apply(lambda s: (s == "Yes").mean() * 100)
      .round(1)
      .sort_values(ascending=False)
)

print("\n=== Resumen de variables numéricas ===")
display(df[["tenure", "MonthlyCharges"]].describe().round(2))

=== Distribución de Churn (variable objetivo) ===


,count
Churn,
No,5174
Yes,1869


,proportion
Churn,
No,73.5
Yes,26.5



=== % de abandono según tipo de contrato ===


,Churn
Contract,
Month-to-month,42.7
One year,11.3
Two year,2.8



=== Resumen de variables numéricas ===


,tenure,MonthlyCharges
count,7043.00,7043.00
mean,32.37,64.76
std,24.56,30.09
min,0.00,18.25
25%,9.00,35.50
50%,29.00,70.35
75%,55.00,89.85
max,72.00,118.75


## Análisis del dataset
- **Unidad de observación:** cada fila es un cliente de una empresa de telecomunicaciones.
- **Objetivo:** predecir si el cliente abandonará el servicio (Churn = Yes/No).
- **Tipo de problema:** clasificación binaria (ruta supervisada).
- **Datos faltantes:** 11 clientes con TotalCharges vacío; todos tienen tenure = 0 (clientes nuevos sin cobro aún).
- **Desbalance:** ~26.5% abandona, por eso usaremos F1 y recall además de accuracy.
- **Se excluye customerID:** es un identificador y no aporta información predictiva.

In [6]:
df_clean = df.copy()

# 1. TotalCharges: de texto a número (los espacios en blanco quedan como NaN)
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# 2. SeniorCitizen viene como 0/1; lo pasamos a "No"/"Yes" para que sea
#    consistente con el resto de variables categóricas del dataset
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 3. Variable objetivo: Churn "Yes" -> 1 (abandona), "No" -> 0 (se queda)
df_clean["Churn"] = df_clean["Churn"].map({"No": 0, "Yes": 1})

print("TotalCharges ahora es:", df_clean["TotalCharges"].dtype)
print("Vacíos en TotalCharges:", df_clean["TotalCharges"].isna().sum())
print("Valores de SeniorCitizen:", df_clean["SeniorCitizen"].unique())
print("Valores de Churn:", df_clean["Churn"].unique())

TotalCharges ahora es: float64
Vacíos en TotalCharges: 11
Valores de SeniorCitizen: ['No' 'Yes']
Valores de Churn: [0 1]


In [7]:
# TotalCharges debería ser aproximadamente tenure x MonthlyCharges
estimado = df_clean["tenure"] * df_clean["MonthlyCharges"]
correlacion = df_clean["TotalCharges"].corr(estimado)

print(f"Correlación entre TotalCharges y (tenure x MonthlyCharges): {correlacion:.4f}")

Correlación entre TotalCharges y (tenure x MonthlyCharges): 0.9996


In [8]:
candidatas = [
    "Contract", "InternetService", "PaymentMethod", "OnlineSecurity",
    "TechSupport", "PaperlessBilling", "SeniorCitizen", "Partner",
    "Dependents", "gender", "PhoneService"
]

for col in candidatas:
    tasa = (df_clean.groupby(col)["Churn"].mean() * 100).round(1)
    print(f"\n=== % de abandono según {col} ===")
    print(tasa.sort_values(ascending=False).to_string())


=== % de abandono según Contract ===
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8

=== % de abandono según InternetService ===
InternetService
Fiber optic    41.9
DSL            19.0
No              7.4

=== % de abandono según PaymentMethod ===
PaymentMethod
Electronic check             45.3
Mailed check                 19.1
Bank transfer (automatic)    16.7
Credit card (automatic)      15.2

=== % de abandono según OnlineSecurity ===
OnlineSecurity
No                     41.8
Yes                    14.6
No internet service     7.4

=== % de abandono según TechSupport ===
TechSupport
No                     41.6
Yes                    15.2
No internet service     7.4

=== % de abandono según PaperlessBilling ===
PaperlessBilling
Yes    33.6
No     16.3

=== % de abandono según SeniorCitizen ===
SeniorCitizen
Yes    41.7
No     23.6

=== % de abandono según Partner ===
Partner
No     33.0
Yes    19.7

=== % de abandono según Dependents ===
Dependents
No

In [9]:
TARGET = "Churn"

numeric_features = [
    "tenure",           # meses que lleva el cliente en la empresa
    "MonthlyCharges"    # cargo mensual
]

categorical_features = [
    "Contract",         # tipo de contrato
    "InternetService",  # tipo de internet
    "PaymentMethod",    # método de pago
    "OnlineSecurity",   # tiene seguridad en línea
    "TechSupport",      # tiene soporte técnico
    "PaperlessBilling", # factura electrónica
    "SeniorCitizen",    # adulto mayor
    "Partner",          # tiene pareja
    "Dependents"        # tiene dependientes
]

features = numeric_features + categorical_features

X = df_clean[features].copy()
y = df_clean[TARGET].copy()

print("Variables del modelo:", len(features))
print("X:", X.shape, "| y:", y.shape)
X.head()

Variables del modelo: 11
X: (7043, 11) | y: (7043,)


,tenure,MonthlyCharges,Contract,InternetService,PaymentMethod,OnlineSecurity,TechSupport,PaperlessBilling,SeniorCitizen,Partner,Dependents
0,1,29.85,Month-to-month,DSL,Electronic check,No,No,Yes,No,Yes,No
1,34,56.95,One year,DSL,Mailed check,Yes,No,No,No,No,No
2,2,53.85,Month-to-month,DSL,Mailed check,Yes,No,Yes,No,No,No
3,45,42.30,One year,DSL,Bank transfer (automatic),Yes,Yes,No,No,No,No
4,2,70.70,Month-to-month,Fiber optic,Electronic check,No,No,Yes,No,No,No


## Selección de variables
**Variables incluidas (11):**
- Numéricas: `tenure`, `MonthlyCharges`.
- Categóricas: `Contract`, `InternetService`, `PaymentMethod`, `OnlineSecurity`, `TechSupport`, `PaperlessBilling`, `SeniorCitizen`, `Partner`, `Dependents`.

**Variables excluidas y por qué:**
- `customerID`: identificador, no tiene valor predictivo.
- `TotalCharges`: correlación ~0.99 con tenure × MonthlyCharges (redundante) y permitiría entradas inconsistentes en la app.
- `gender`, `PhoneService`: la tasa de abandono casi no cambia entre sus categorías.

**Limpieza:** TotalCharges convertido a numérico; SeniorCitizen pasado a Yes/No; Churn codificado como 1 (abandona) / 0 (se queda).

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # 20% para prueba
    random_state=42,    # semilla: siempre la misma división
    stratify=y          # mantiene el 73.5% / 26.5% en ambos grupos
)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("% abandono en train:", round(y_train.mean() * 100, 1))
print("% abandono en test: ", round(y_test.mean() * 100, 1))

Entrenamiento: (5634, 11) | Prueba: (1409, 11)
% abandono en train: 26.5
% abandono en test:  26.5


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score

# --- Preprocesamiento ---
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # rellena vacíos con la mediana
    ("scaler", StandardScaler())                     # pone las numéricas en la misma escala
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),  # rellena vacíos con la categoría más común
    ("onehot", OneHotEncoder(handle_unknown="ignore"))     # convierte categorías en columnas 0/1
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

# --- Modelos a comparar ---
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Regresión logística balanceada": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random forest balanceado": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=10,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
}

resultados = []
pipelines = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    pipelines[nombre] = pipe
    resultados.append({
        "Modelo": nombre,
        "F1 (abandona)": f1_score(y_test, pred),
        "Recall (abandona)": recall_score(y_test, pred),
        "Precisión (abandona)": precision_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "Accuracy": accuracy_score(y_test, pred),
    })

tabla = pd.DataFrame(resultados).set_index("Modelo").round(3)
display(tabla)

mejor_nombre = tabla["F1 (abandona)"].idxmax()
model = pipelines[mejor_nombre]
print(f"\n🏆 Mejor modelo según F1: {mejor_nombre}")

,F1 (abandona),Recall (abandona),Precisión (abandona),ROC-AUC,Accuracy
Modelo,,,,,
Regresión logística,0.583,0.532,0.644,0.836,0.798
Regresión logística balanceada,0.618,0.797,0.505,0.836,0.739
Random forest balanceado,0.623,0.754,0.530,0.840,0.757



🏆 Mejor modelo según F1: Random forest balanceado


In [12]:
from sklearn.metrics import confusion_matrix, classification_report

pred = model.predict(X_test)

cm = pd.DataFrame(
    confusion_matrix(y_test, pred),
    index=["Real: se queda", "Real: abandona"],
    columns=["Predicho: se queda", "Predicho: abandona"]
)
print(f"=== Matriz de confusión — {mejor_nombre} ===")
display(cm)

print("\n=== Reporte de clasificación ===")
print(classification_report(y_test, pred, target_names=["Se queda", "Abandona"]))

=== Matriz de confusión — Random forest balanceado ===


,Predicho: se queda,Predicho: abandona
Real: se queda,785,250
Real: abandona,92,282



=== Reporte de clasificación ===
              precision    recall  f1-score   support

    Se queda       0.90      0.76      0.82      1035
    Abandona       0.53      0.75      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [13]:
import joblib

# Opciones válidas de cada categórica (para los selectbox de la app)
category_options = {
    col: sorted(X_train[col].dropna().astype(str).unique().tolist())
    for col in categorical_features
}

# Rangos de las numéricas (para validar entradas en la app)
numeric_ranges = {
    col: {
        "min": float(X_train[col].min()),
        "max": float(X_train[col].max()),
        "median": float(X_train[col].median())
    }
    for col in numeric_features
}

artifact = {
    "model": model,
    "model_name": mejor_nombre,
    "features": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "category_options": category_options,
    "numeric_ranges": numeric_ranges,
    "target_labels": {0: "Se queda", 1: "Abandona"},
    "task": "classification",
    "metrics": tabla.loc[mejor_nombre].to_dict(),
}

joblib.dump(artifact, PROJECT / "model.joblib")

tamano_mb = (PROJECT / "model.joblib").stat().st_size / 1024 / 1024
print(f"✅ Modelo guardado en {PROJECT / 'model.joblib'} ({tamano_mb:.2f} MB)")

✅ Modelo guardado en /content/ml_streamlit_project/model.joblib (7.05 MB)


## Modelo y evaluación
Se compararon 3 modelos dentro del mismo Pipeline (preprocesamiento + clasificador):
regresión logística, regresión logística balanceada y random forest balanceado.

**Modelo elegido: Random Forest balanceado** (mayor F1 para la clase "abandona").

| Métrica (clase "abandona") | Valor |
|---|---|
| Recall | 0.75 |
| Precisión | 0.53 |
| F1 | 0.62 |
| F1 weighted | 0.77 |
| Accuracy | 0.76 |

**¿Por qué F1 y recall?** Las clases están desbalanceadas (26.5% abandona). Un modelo que siempre
dijera "se queda" tendría 73.5% de accuracy sin detectar a nadie. Para el negocio es más costoso
no detectar a un cliente que se va (92 casos en prueba) que dar una falsa alarma (250 casos),
por eso priorizamos recall y elegimos el modelo con mejor F1.

**Matriz de confusión (prueba, 1409 clientes):** 282 abandonos detectados de 374.

In [14]:
%%writefile /content/ml_streamlit_project/train.py
"""
Entrenamiento del modelo de abandono de clientes (Telco Customer Churn).

Uso:
    python train.py

Lee data/telco_churn.csv, limpia los datos, entrena un Pipeline
(preprocesamiento + Random Forest balanceado), lo evalúa y guarda
todo lo necesario para la app en model.joblib.
"""
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ----------------------------------------------------------------------
# Configuración
# ----------------------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "data" / "telco_churn.csv"
MODEL_PATH = BASE_DIR / "model.joblib"
RANDOM_STATE = 42

TARGET = "Churn"
NUMERIC_FEATURES = ["tenure", "MonthlyCharges"]
CATEGORICAL_FEATURES = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport",
    "PaperlessBilling",
    "SeniorCitizen",
    "Partner",
    "Dependents",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


# ----------------------------------------------------------------------
# Datos
# ----------------------------------------------------------------------
def load_data() -> pd.DataFrame:
    """Carga el dataset crudo desde data/."""
    return pd.read_csv(DATA_PATH)


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica la misma limpieza validada en el notebook."""
    df = df.copy()
    # TotalCharges viene como texto; 11 clientes con tenure=0 lo tienen vacío
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    # SeniorCitizen 0/1 -> No/Yes para que sea consistente con las demás categóricas
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
    # Objetivo: 1 = abandona, 0 = se queda
    df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})
    return df


# ----------------------------------------------------------------------
# Modelo
# ----------------------------------------------------------------------
def build_pipeline() -> Pipeline:
    """Preprocesamiento + clasificador en un solo objeto."""
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])
    classifier = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])


def train_model(df: pd.DataFrame) -> dict:
    """Entrena, evalúa y construye el artefacto para la app."""
    X = df[FEATURES].copy()
    y = df[TARGET].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    model = build_pipeline()
    model.fit(X_train, y_train)

    # Evaluación
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "f1": round(f1_score(y_test, pred), 3),
        "recall": round(recall_score(y_test, pred), 3),
        "precision": round(precision_score(y_test, pred), 3),
        "roc_auc": round(roc_auc_score(y_test, proba), 3),
        "accuracy": round(accuracy_score(y_test, pred), 3),
    }

    print("=== Matriz de confusión (filas = real, columnas = predicho) ===")
    print(confusion_matrix(y_test, pred))
    print("\n=== Reporte de clasificación ===")
    print(classification_report(y_test, pred, target_names=["Se queda", "Abandona"]))
    print("=== Métricas clase 'Abandona' ===")
    for name, value in metrics.items():
        print(f"{name:>10}: {value}")

    # Información para construir la interfaz
    category_options = {
        col: sorted(X_train[col].dropna().astype(str).unique().tolist())
        for col in CATEGORICAL_FEATURES
    }
    numeric_ranges = {
        col: {
            "min": float(X_train[col].min()),
            "max": float(X_train[col].max()),
            "median": float(X_train[col].median()),
        }
        for col in NUMERIC_FEATURES
    }

    return {
        "model": model,
        "model_name": "Random Forest balanceado",
        "features": FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "category_options": category_options,
        "numeric_ranges": numeric_ranges,
        "target_labels": {0: "Se queda", 1: "Abandona"},
        "task": "classification",
        "metrics": metrics,
    }


def main() -> None:
    df = clean_data(load_data())
    artifact = train_model(df)
    joblib.dump(artifact, MODEL_PATH)
    print(f"\nModelo guardado en {MODEL_PATH}")


if __name__ == "__main__":
    main()

Writing /content/ml_streamlit_project/train.py


In [15]:
!cd /content/ml_streamlit_project && python train.py

=== Matriz de confusión (filas = real, columnas = predicho) ===
[[785 250]
 [ 92 282]]

=== Reporte de clasificación ===
              precision    recall  f1-score   support

    Se queda       0.90      0.76      0.82      1035
    Abandona       0.53      0.75      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409

=== Métricas clase 'Abandona' ===
        f1: 0.623
    recall: 0.754
 precision: 0.53
   roc_auc: 0.84
  accuracy: 0.757

Modelo guardado en /content/ml_streamlit_project/model.joblib


In [20]:
%%writefile /content/ml_streamlit_project/app.py
"""
App de Streamlit para predecir el abandono de clientes (Telco Customer Churn).

Uso:
    streamlit run app.py

Carga model.joblib (generado por train.py) y NO vuelve a entrenar.
"""
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "model.joblib"

# Etiquetas en español para la interfaz (el modelo recibe los valores originales)
FEATURE_LABELS = {
    "tenure": "Meses como cliente",
    "MonthlyCharges": "Cargo mensual (USD)",
    "Contract": "Tipo de contrato",
    "InternetService": "Servicio de internet",
    "PaymentMethod": "Método de pago",
    "OnlineSecurity": "Seguridad en línea",
    "TechSupport": "Soporte técnico",
    "PaperlessBilling": "Factura electrónica",
    "SeniorCitizen": "Adulto mayor (65+)",
    "Partner": "Tiene pareja",
    "Dependents": "Tiene personas a cargo",
}

VALUE_LABELS = {
    "Yes": "Sí",
    "No": "No",
    "No internet service": "Sin servicio de internet",
    "Month-to-month": "Mes a mes",
    "One year": "Un año",
    "Two year": "Dos años",
    "DSL": "DSL",
    "Fiber optic": "Fibra óptica",
    "Electronic check": "Cheque electrónico",
    "Mailed check": "Cheque por correo",
    "Bank transfer (automatic)": "Transferencia bancaria (automática)",
    "Credit card (automatic)": "Tarjeta de crédito (automática)",
}

# Variables que dependen de tener internet
INTERNET_DEPENDENT = ["OnlineSecurity", "TechSupport"]


def label(value: str) -> str:
    return VALUE_LABELS.get(value, value)


@st.cache_resource
def load_artifact():
    """Carga el modelo una sola vez (no reentrena)."""
    return joblib.load(MODEL_PATH)


st.set_page_config(page_title="Predicción de abandono", page_icon="📡", layout="centered")

artifact = load_artifact()
model = artifact["model"]
features = artifact["features"]
numeric_features = artifact["numeric_features"]
categorical_features = artifact["categorical_features"]
category_options = artifact["category_options"]
numeric_ranges = artifact["numeric_ranges"]
metrics = artifact.get("metrics", {})

# ----------------------------------------------------------------------
# Panel lateral: información del modelo
# ----------------------------------------------------------------------
with st.sidebar:
    st.header("ℹ️ Sobre el modelo")
    st.write(f"**Modelo:** {artifact.get('model_name', 'Clasificador')}")
    st.write("**Dataset:** Telco Customer Churn (IBM), 7.043 clientes.")
    st.write("**Objetivo:** predecir si un cliente abandonará el servicio.")
    if metrics:
        st.subheader("Desempeño en prueba")
        st.metric("Recall (abandona)", metrics.get("recall"))
        st.metric("F1 (abandona)", metrics.get("f1"))
        st.metric("ROC-AUC", metrics.get("roc_auc"))
    st.caption("Recall: proporción de clientes que abandonan y el modelo detecta.")

# ----------------------------------------------------------------------
# Formulario
# ----------------------------------------------------------------------
st.title("📡 Predicción de abandono de clientes")
st.write(
    "Ingrese las características de un cliente de telecomunicaciones para estimar "
    "la probabilidad de que **cancele el servicio**."
)

values = {}

st.subheader("Cuenta y facturación")
col1, col2 = st.columns(2)
with col1:
    r = numeric_ranges["tenure"]
    values["tenure"] = st.slider(
        FEATURE_LABELS["tenure"],
        min_value=int(r["min"]), max_value=int(r["max"]), value=int(r["median"]),
        help="Cantidad de meses que el cliente lleva en la empresa.",
    )
with col2:
    r = numeric_ranges["MonthlyCharges"]
    values["MonthlyCharges"] = st.number_input(
        FEATURE_LABELS["MonthlyCharges"],
        min_value=float(r["min"]), max_value=float(r["max"]),
        value=float(r["median"]), step=0.5,
        help="Valor que paga el cliente cada mes.",
    )

col1, col2 = st.columns(2)
with col1:
    values["Contract"] = st.selectbox(
        FEATURE_LABELS["Contract"], category_options["Contract"], format_func=label
    )
    values["PaperlessBilling"] = st.selectbox(
        FEATURE_LABELS["PaperlessBilling"], category_options["PaperlessBilling"], format_func=label
    )
with col2:
    values["PaymentMethod"] = st.selectbox(
        FEATURE_LABELS["PaymentMethod"], category_options["PaymentMethod"], format_func=label
    )

st.subheader("Servicios")
values["InternetService"] = st.selectbox(
    FEATURE_LABELS["InternetService"], category_options["InternetService"], format_func=label
)

has_internet = values["InternetService"] != "No"
col1, col2 = st.columns(2)
for col, feature in zip([col1, col2], INTERNET_DEPENDENT):
    with col:
        if has_internet:
            options = [o for o in category_options[feature] if o != "No internet service"]
        else:
            options = ["No internet service"]
        values[feature] = st.selectbox(
            FEATURE_LABELS[feature], options, format_func=label, disabled=not has_internet
        )

st.subheader("Perfil del cliente")
col1, col2, col3 = st.columns(3)
for col, feature in zip([col1, col2, col3], ["SeniorCitizen", "Partner", "Dependents"]):
    with col:
        values[feature] = st.selectbox(
            FEATURE_LABELS[feature], category_options[feature], format_func=label
        )

# ----------------------------------------------------------------------
# Predicción
# ----------------------------------------------------------------------
st.divider()
if st.button("🔮 Predecir abandono", type="primary", use_container_width=True):
    X_new = pd.DataFrame([values], columns=features)

    pred = int(model.predict(X_new)[0])
    proba = float(model.predict_proba(X_new)[0][1])

    if proba < 0.30:
        nivel, mensaje = "🟢 Riesgo BAJO", "El cliente probablemente se quedará."
    elif proba < 0.60:
        nivel, mensaje = "🟡 Riesgo MEDIO", "Conviene hacer seguimiento al cliente."
    else:
        nivel, mensaje = "🔴 Riesgo ALTO", "Se recomienda una acción de retención (oferta, cambio de contrato, contacto)."

    st.subheader("Resultado")
    c1, c2 = st.columns(2)
    porcentaje = f"{proba * 100:.1f}%"
    c1.metric("Probabilidad de abandono", porcentaje)
    c2.metric("Predicción del modelo", artifact["target_labels"][pred])
    st.progress(proba)

Overwriting /content/ml_streamlit_project/app.py


In [18]:
import subprocess, time

# Cierra cualquier Streamlit que ya esté corriendo
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
time.sleep(1)

# Arranca Streamlit en segundo plano, sin pedir correo (headless)
log = open("/content/streamlit.log", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    cwd="/content/ml_streamlit_project",
    stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL
)

time.sleep(8)
print(open("/content/streamlit.log").read())



2026-11-11 02:12:50.526 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.143.141.97:8501




In [22]:
!npx -y localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://chilly-grapes-turn.loca.lt
^C


In [23]:
!cd /content/ml_streamlit_project && python -m py_compile app.py && echo "✅ app.py sin errores de sintaxis"

✅ app.py sin errores de sintaxis


In [24]:
import joblib
import pandas as pd

art = joblib.load("/content/ml_streamlit_project/model.joblib")

casos = pd.DataFrame([
    {   # Caso 1: riesgo alto
        "tenure": 2, "MonthlyCharges": 90.0, "Contract": "Month-to-month",
        "InternetService": "Fiber optic", "PaymentMethod": "Electronic check",
        "OnlineSecurity": "No", "TechSupport": "No", "PaperlessBilling": "Yes",
        "SeniorCitizen": "Yes", "Partner": "No", "Dependents": "No",
    },
    {   # Caso 2: riesgo bajo
        "tenure": 60, "MonthlyCharges": 25.0, "Contract": "Two year",
        "InternetService": "No", "PaymentMethod": "Bank transfer (automatic)",
        "OnlineSecurity": "No internet service", "TechSupport": "No internet service",
        "PaperlessBilling": "No", "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "Yes",
    },
    {   # Caso 3: intermedio
        "tenure": 20, "MonthlyCharges": 70.0, "Contract": "One year",
        "InternetService": "DSL", "PaymentMethod": "Credit card (automatic)",
        "OnlineSecurity": "Yes", "TechSupport": "No", "PaperlessBilling": "Yes",
        "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "No",
    },
], columns=art["features"])

pred = art["model"].predict(casos)
proba = art["model"].predict_proba(casos)[:, 1]

for i, (p, c) in enumerate(zip(proba, pred), start=1):
    print(f"Caso {i}: probabilidad de abandono = {p:.1%} → {art['target_labels'][int(c)]}")

Caso 1: probabilidad de abandono = 91.3% → Abandona
Caso 2: probabilidad de abandono = 0.9% → Se queda
Caso 3: probabilidad de abandono = 19.4% → Se queda


In [25]:
import sklearn, pandas, numpy, joblib, streamlit, openpyxl

requirements = f"""streamlit=={streamlit.__version__}
pandas=={pandas.__version__}
numpy=={numpy.__version__}
scikit-learn=={sklearn.__version__}
joblib=={joblib.__version__}
openpyxl=={openpyxl.__version__}
"""

with open("/content/ml_streamlit_project/requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)

streamlit==1.67.0
pandas==2.2.3
numpy==2.4.6
scikit-learn==1.6.1
joblib==1.6.0
openpyxl==3.1.5



In [26]:
%%writefile /content/ml_streamlit_project/.dockerignore
.git
.venv
__pycache__
*.pyc
.ipynb_checkpoints
.env
*.log
*.zip

Writing /content/ml_streamlit_project/.dockerignore


In [27]:
%%writefile /content/ml_streamlit_project/Dockerfile
# Imagen base: Python 3.13 (misma versión usada para entrenar en Colab)
FROM python:3.13-slim

# Evita archivos .pyc y muestra los logs en tiempo real
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# 1) Primero solo las dependencias (aprovecha la caché de capas de Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 2) Luego el resto del proyecto (app.py, model.joblib, train.py, data/...)
COPY . .

# Puerto en el que escucha Streamlit
EXPOSE 8501

# Docker revisa periódicamente que la app responda
HEALTHCHECK --interval=30s --timeout=5s --start-period=20s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8501/_stcore/health')" || exit 1

# Proceso que se ejecuta al iniciar el contenedor
CMD ["streamlit", "run", "app.py", \
     "--server.address=0.0.0.0", \
     "--server.port=8501", \
     "--server.headless=true"]

Writing /content/ml_streamlit_project/Dockerfile


In [29]:
GITHUB_USER = "hudynikolasbarrera-ship-it"
AUTOR = "Nicolás Barrera Castañeda"
REPO = "telco-churn-streamlit"

readme = """# 📡 Predicción de abandono de clientes (Telco Customer Churn)

Solución completa de Machine Learning: desde el dataset hasta el despliegue en
Google Cloud. Incluye entrenamiento reproducible (`train.py`), una app web en
Streamlit (`app.py`), empaquetado con Docker y despliegue en una VM de
Google Compute Engine.

## 1. Objetivo
Predecir si un cliente de una empresa de telecomunicaciones **abandonará el
servicio (churn)**, para poder aplicar acciones de retención a tiempo.

## 2. Dataset
- **Fuente:** Telco Customer Churn (IBM), dataset público.
  URL original: https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
- **Ubicación en el repo:** `data/telco_churn.csv`
- **Tamaño:** 7.043 filas × 21 columnas.
- **Unidad de observación:** cada fila es **un cliente**.
- **Variable objetivo:** `Churn` (Yes = abandona, No = se queda).
- **Desbalance:** 73,5 % se queda / 26,5 % abandona.

## 3. Ruta elegida
**Ruta A – Aprendizaje supervisado:** clasificación binaria.

## 4. Variables utilizadas (11)
| Tipo | Variable | Descripción |
|---|---|---|
| Numérica | `tenure` | Meses como cliente |
| Numérica | `MonthlyCharges` | Cargo mensual (USD) |
| Categórica | `Contract` | Tipo de contrato |
| Categórica | `InternetService` | Tipo de servicio de internet |
| Categórica | `PaymentMethod` | Método de pago |
| Categórica | `OnlineSecurity` | Tiene seguridad en línea |
| Categórica | `TechSupport` | Tiene soporte técnico |
| Categórica | `PaperlessBilling` | Factura electrónica |
| Categórica | `SeniorCitizen` | Adulto mayor (65+) |
| Categórica | `Partner` | Tiene pareja |
| Categórica | `Dependents` | Tiene personas a cargo |

**Variables excluidas:**
- `customerID`: identificador, sin valor predictivo.
- `TotalCharges`: correlación ≈ 0,99 con `tenure × MonthlyCharges` (redundante)
  y permitiría entradas inconsistentes en la app.
- `gender`, `PhoneService`: la tasa de abandono casi no cambia entre sus categorías.

**Hallazgo clave de la exploración:** el abandono es de 42,7 % en contratos
mes a mes, 11,3 % en contratos de un año y 2,8 % en contratos de dos años.

## 5. Preparación de datos
- `TotalCharges` venía como texto: 11 clientes con `tenure = 0` (clientes nuevos
  sin cobro aún) lo tenían vacío. Se convirtió a numérico.
- `SeniorCitizen` (0/1) se pasó a `No`/`Yes`, consistente con las demás categóricas.
- `Churn` se codificó como 1 (abandona) / 0 (se queda).
- División 80/20 estratificada (`random_state=42`).
- Todo el preprocesamiento está **dentro del Pipeline** de scikit-learn:
  - Numéricas: imputación por mediana + `StandardScaler`.
  - Categóricas: imputación por moda + `OneHotEncoder(handle_unknown="ignore")`.

Así, la app aplica exactamente las mismas transformaciones que el entrenamiento.

## 6. Modelo y justificación
Se compararon tres modelos dentro del mismo Pipeline:

| Modelo | Comentario |
|---|---|
| Regresión logística | Línea base |
| Regresión logística balanceada | `class_weight="balanced"` |
| **Random Forest balanceado** ✅ | Mejor F1 en la clase "abandona" |

Modelo final: `RandomForestClassifier(n_estimators=300, max_depth=10,
min_samples_leaf=10, class_weight="balanced", random_state=42)`.
`class_weight="balanced"` compensa el desbalance; `max_depth` y
`min_samples_leaf` limitan el sobreajuste y el tamaño del archivo.

## 7. Evaluación (conjunto de prueba, 1.409 clientes)
| Métrica (clase "abandona") | Valor |
|---|---|
| Recall | 0,75 |
| Precisión | 0,53 |
| F1 | 0,62 |
| ROC-AUC | 0,84 |
| Accuracy | 0,76 |
| F1 weighted | 0,77 |

**Matriz de confusión:**

| | Predicho: se queda | Predicho: abandona |
|---|---|---|
| **Real: se queda** | 785 | 250 |
| **Real: abandona** | 92 | 282 |

**¿Por qué F1 y recall y no solo accuracy?** Con 26,5 % de abandono, un modelo
que siempre dijera "se queda" tendría 73,5 % de accuracy sin detectar a nadie.
Para el negocio es más costoso no detectar a un cliente que se va (falso
negativo) que una falsa alarma (falso positivo), por eso se prioriza recall y
se elige el modelo con mejor F1. El modelo detecta 282 de 374 abandonos (75 %).

## 8. Interpretación de la salida de la app
La app muestra:
- **Probabilidad de abandono** (0–100 %).
- **Predicción:** "Abandona" si la probabilidad ≥ 50 %, si no "Se queda".
- **Nivel de riesgo:** 🟢 bajo (< 30 %), 🟡 medio (30–60 %), 🔴 alto (≥ 60 %),
  con una recomendación de acción.

Si el cliente **no tiene internet**, los campos "Seguridad en línea" y
"Soporte técnico" se fijan automáticamente en "Sin servicio de internet"
para evitar combinaciones imposibles.

## 9. Estructura del proyecto
~~~
{{REPO}}/
├── app.py              # App Streamlit (carga el modelo, NO reentrena)
├── train.py            # Entrenamiento reproducible → model.joblib
├── model.joblib        # Pipeline entrenado + metadatos para la interfaz
├── requirements.txt    # Dependencias con versiones fijas
├── Dockerfile          # Construcción de la imagen
├── .dockerignore       # Archivos excluidos de la imagen
├── README.md
└── data/
    └── telco_churn.csv # Dataset
~~~

## 10. Cómo ejecutar `train.py` (reentrenar el modelo)
Requiere Python 3.13.
~~~bash
pip install -r requirements.txt
python train.py
~~~
Salida esperada: matriz de confusión, reporte de clasificación, métricas y
`Modelo guardado en .../model.joblib`.

## 11. Cómo ejecutar Streamlit (sin Docker)
~~~bash
pip install -r requirements.txt
streamlit run app.py
~~~
Abrir http://localhost:8501

## 12. Cómo construir la imagen Docker
Desde la carpeta del proyecto:
~~~bash
docker build -t ml-streamlit-app .
docker images
~~~

## 13. Cómo ejecutar el contenedor
Prueba interactiva (se detiene con Ctrl+C):
~~~bash
docker run --rm -p 8501:8501 ml-streamlit-app
~~~
En segundo plano (modo despliegue):
~~~bash
docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
docker ps
docker logs ml-streamlit
~~~
Abrir `http://localhost:8501` (local) o `http://IP_EXTERNA:8501` (VM).

## 14. Despliegue en Google Cloud (Compute Engine)
1. Crear una VM **Ubuntu 24.04 LTS** (por ejemplo `e2-small`) con IP externa
   y la etiqueta de red `streamlit-server`.
2. Abrir la terminal con el botón **SSH** de la consola e instalar Docker y Git:
~~~bash
sudo apt update
sudo apt install -y docker.io git
sudo systemctl enable --now docker
~~~
3. Clonar y desplegar:
~~~bash
git clone https://github.com/{{USER}}/{{REPO}}.git
cd {{REPO}}
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
~~~
4. Crear la regla de firewall (desde Cloud Shell):
~~~bash
gcloud compute firewall-rules create allow-streamlit-8501 --allow=tcp:8501 --direction=INGRESS --target-tags=streamlit-server
~~~
5. Abrir `http://IP_EXTERNA_DE_LA_VM:8501`

**Actualizar la app después de cambios en GitHub:**
~~~bash
git pull
sudo docker stop ml-streamlit && sudo docker rm ml-streamlit
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
~~~

⚠️ Al terminar, **detener o eliminar la VM** para no consumir créditos.

## 15. Ejemplos de entrada y salida
| Variable | Caso 1 | Caso 2 | Caso 3 |
|---|---|---|---|
| Meses como cliente | 2 | 60 | 20 |
| Cargo mensual | 90 | 25 | 70 |
| Contrato | Mes a mes | Dos años | Un año |
| Internet | Fibra óptica | No | DSL |
| Método de pago | Cheque electrónico | Transferencia (automática) | Tarjeta de crédito (automática) |
| Seguridad en línea | No | Sin servicio de internet | Sí |
| Soporte técnico | No | Sin servicio de internet | No |
| Factura electrónica | Sí | No | Sí |
| Adulto mayor | Sí | No | No |
| Tiene pareja | No | Sí | Sí |
| Personas a cargo | No | Sí | No |
| **Probabilidad de abandono** | **91,3 %** | **0,9 %** | **19,4 %** |
| **Predicción** | 🔴 Abandona | 🟢 Se queda | 🟢 Se queda |

## 16. Solución de problemas
| Situación | Revisión |
|---|---|
| `docker build` falla | Leer el error completo; revisar `requirements.txt` y `Dockerfile`. |
| El contenedor se detiene | `sudo docker ps -a` y `sudo docker logs ml-streamlit`. |
| Funciona en la VM pero no desde fuera | Revisar regla de firewall, etiqueta `streamlit-server` e IP externa. |
| Cambios no aparecen | `git pull` → detener/eliminar contenedor → `docker build` → `docker run`. |

## 17. Autor
- {{AUTOR}}

Machine Learning 2
"""

readme = (readme.replace("{{USER}}", GITHUB_USER)
                .replace("{{REPO}}", REPO)
                .replace("{{AUTOR}}", AUTOR))

with open("/content/ml_streamlit_project/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("✅ README.md creado")
print(f"   Repo esperado: https://github.com/{GITHUB_USER}/{REPO}")

✅ README.md creado
   Repo esperado: https://github.com/hudynikolasbarrera-ship-it/telco-churn-streamlit


In [30]:
!cd /content/ml_streamlit_project && ls -la && echo "--- data/ ---" && ls -la data

total 7280
drwxr-xr-x 4 root root    4096 Nov 11 02:31 .
drwxr-xr-x 1 root root    4096 Nov 11 02:06 ..
-rw-r--r-- 1 root root    6334 Nov 11 02:19 app.py
drwxr-xr-x 2 root root    4096 Nov 11 01:40 data
-rw-r--r-- 1 root root     980 Nov 11 02:26 Dockerfile
-rw-r--r-- 1 root root      65 Nov 11 02:26 .dockerignore
-rw-r--r-- 1 root root 7396917 Nov 11 02:05 model.joblib
drwxr-xr-x 2 root root    4096 Nov 11 02:24 __pycache__
-rw-r--r-- 1 root root    8655 Nov 11 02:31 README.md
-rw-r--r-- 1 root root      95 Nov 11 02:25 requirements.txt
-rw-r--r-- 1 root root    5499 Nov 11 02:03 train.py
--- data/ ---
total 960
drwxr-xr-x 2 root root   4096 Nov 11 01:40 .
drwxr-xr-x 4 root root   4096 Nov 11 02:31 ..
-rw-r--r-- 1 root root 971241 Nov 11 01:40 telco_churn.csv


In [31]:
!cd /content && rm -f ml_streamlit_project.zip && zip -r ml_streamlit_project.zip ml_streamlit_project -x "*/__pycache__/*" "*/.ipynb_checkpoints/*"

from google.colab import files
files.download("/content/ml_streamlit_project.zip")

  adding: ml_streamlit_project/ (stored 0%)
  adding: ml_streamlit_project/Dockerfile (deflated 37%)
  adding: ml_streamlit_project/README.md (deflated 56%)
  adding: ml_streamlit_project/train.py (deflated 63%)
  adding: ml_streamlit_project/model.joblib (deflated 61%)
  adding: ml_streamlit_project/.dockerignore (deflated 6%)
  adding: ml_streamlit_project/requirements.txt (deflated 14%)
  adding: ml_streamlit_project/app.py (deflated 63%)
  adding: ml_streamlit_project/data/ (stored 0%)
  adding: ml_streamlit_project/data/telco_churn.csv (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import os
from streamlit.testing.v1 import AppTest

os.chdir("/content/ml_streamlit_project")
at = AppTest.from_file("/content/ml_streamlit_project/app.py", default_timeout=60)

def sel(label):
    """Busca un selectbox por su etiqueta."""
    return next(s for s in at.selectbox if s.label == label)

def reporte(titulo):
    print(f"\n=== {titulo} ===")
    print("Errores:", len(at.exception))
    for e in at.exception:
        print("  ❌", e.value)
    for w in at.warning:
        print("  ⚠️", w.value)

# ---------- Prueba 1: la app carga ----------
at.run()
reporte("1) Carga inicial de la app")

# ---------- Prueba 2: Caso 1 (riesgo alto) y botón ----------
at.slider[0].set_value(2)
at.number_input[0].set_value(90.0)
sel("Tipo de contrato").set_value("Month-to-month")
sel("Factura electrónica").set_value("Yes")
sel("Método de pago").set_value("Electronic check")
sel("Servicio de internet").set_value("Fiber optic")
at.run()
sel("Seguridad en línea").set_value("No")
sel("Soporte técnico").set_value("No")
sel("Adulto mayor (65+)").set_value("Yes")
sel("Tiene pareja").set_value("No")
sel("Tiene personas a cargo").set_value("No")
at.button[0].click().run()

reporte("2) Caso 1 + botón Predecir")
for m in at.metric:
    print(f"  📊 {m.label}: {m.value}")
for msg in list(at.error) + list(at.success):
    print("  💬", msg.value)

# ---------- Prueba 3: validación "sin internet" ----------
sel("Servicio de internet").set_value("No")
at.run()
seg = sel("Seguridad en línea")
sop = sel("Soporte técnico")
reporte("3) Cliente sin internet")
print("  Opciones Seguridad en línea:", seg.options, "| bloqueado:", seg.disabled)
print("  Opciones Soporte técnico:  ", sop.options, "| bloqueado:", sop.disabled)

2026-11-11 02:48:22.881 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.



=== 1) Carga inicial de la app ===
Errores: 0

=== 2) Caso 1 + botón Predecir ===
Errores: 0
  📊 Probabilidad de abandono: 91.3%
  📊 Predicción del modelo: Abandona
  📊 Recall (abandona): 0.754
  📊 F1 (abandona): 0.623
  📊 ROC-AUC: 0.84

=== 3) Cliente sin internet ===
Errores: 0
  Opciones Seguridad en línea: ['Sin servicio de internet'] | bloqueado: True
  Opciones Soporte técnico:   ['Sin servicio de internet'] | bloqueado: True
